# E0.3 — State-space Bollinger bands (SMA → EWMA → LLT → jump-adaptive LLT)

**Question:** can a state-space band react faster to sudden price changes **without** giving up containment?

**Causality (same as E0.2a):** everything is stamped at the **close** of a completed candle. For every method the band for bar *t* is the one **published at the close of t−1**:

`z[t] = (close[t] − center_fc[t]) / scale_fc[t]`, and "inside the k-band" ⇔ `|z[t]| ≤ k`.

This is stricter than the usual Bollinger check, which tests the close against a band that already includes that close (so containment is partly built in).

**Method ladder** (each step adds one idea, so its value can be measured on its own):

| Method | Center | Adds |
|---|---|---|
| `BB(N)` | SMA | classic |
| `EWMA(N)` | EWMA = steady-state local-level Kalman | weighting only |
| `LLT(N)` | local linear trend Kalman (level + slope) | removes trend lag |
| `JA-LLT(N)` | LLT whose level noise is boosted by **P(jump)** from a 2-state Markov-switching filter on returns | re-anchors in 1–2 bars after a jump, smooth otherwise |

**Fair comparison:** containment depends on k, so for every method k is **calibrated on TRAIN (H1 2025) to hit 95% containment**, then frozen. On TEST (H2 2025) we compare containment, band width, reaction after shocks, and forward returns after band breaks, net of fees.

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

import ss_bands as sb

DATA_SOURCE = "binance"      # "binance" (your cache/API) or "synthetic" (code check only)

START = pd.Timestamp("2024-12-30", tz="UTC")   # warm-up before 2025
END = pd.Timestamp("2026-01-02", tz="UTC")     # exclusive
EVAL_START = pd.Timestamp("2025-01-01", tz="UTC")
TRAIN_END = pd.Timestamp("2025-07-01", tz="UTC")   # train = H1, test = H2
EVAL_END = pd.Timestamp("2026-01-01", tz="UTC")
NOV_START, NOV_END = "2025-11-13", "2025-11-23"

TARGET_CONTAINMENT = 0.95
SHOCK_SIGMA = 5.0          # shock = |5m log return| > 5 x EWMA std known at t-1
HORIZONS = [6, 12, 24]     # forward-return horizons in bars (30m, 1h, 2h)
COST_BPS = 20.0            # round trip: 0.10% taker each side; use 15 with BNB discount

REPO = Path("/media/ata/Data/personal/binbot")
DATA_DIR = REPO / "research" / "sentiment_price" / "data"
CACHE_DIR = DATA_DIR / "btcusdt_5m_batches"
STEP_MS = 5 * 60 * 1000
START_MS = int(START.timestamp() * 1000)
END_MS = int(END.timestamp() * 1000)

ModuleNotFoundError: No module named 'research'

## 1. Data (your E0.2a loader, unchanged, wrapped in a function)

In [ ]:
def load_binance_5m():
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    all_candles = []
    cursor = START_MS
    with requests.Session() as session:
        while cursor < END_MS:
            batch_end = min(cursor + 1000 * STEP_MS, END_MS)
            cache_file = CACHE_DIR / f"{cursor}_{batch_end}.json"
            if cache_file.exists():
                batch = json.loads(cache_file.read_text())
            else:
                response = session.get(
                    "https://api.binance.com/api/v3/klines",
                    params={"symbol": "BTCUSDT", "interval": "5m", "timeZone": "0",
                            "startTime": cursor, "endTime": batch_end - 1, "limit": 1000},
                    timeout=30,
                )
                response.raise_for_status()
                batch = response.json()
            if not isinstance(batch, list) or not batch:
                raise ValueError(f"Unexpected response at {cursor}: {batch}")
            expected_opens = list(range(cursor, batch_end, STEP_MS))
            actual_opens = [int(row[0]) for row in batch]
            if actual_opens != expected_opens:
                raise ValueError(f"Missing, duplicate or unexpected candles in batch {cursor}")
            if not cache_file.exists():
                temporary = cache_file.with_suffix(".tmp")
                temporary.write_text(json.dumps(batch))
                temporary.replace(cache_file)
                time.sleep(0.2)
            all_candles.extend(batch)
            cursor = batch_end
            print(f"\rLoaded {len(all_candles):,} candles", end="", flush=True)
    print()
    columns = ["open_time", "open", "high", "low", "close", "volume", "close_time",
               "quote_volume", "trade_count", "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
    bars = pd.DataFrame(all_candles, columns=columns)
    bars["open_time"] = pd.to_datetime(bars["open_time"], unit="ms", utc=True)
    bars["close_time"] = pd.to_datetime(bars["close_time"], unit="ms", utc=True)
    numeric = ["open", "high", "low", "close", "volume", "quote_volume", "trade_count",
               "taker_buy_volume", "taker_buy_quote_volume"]
    bars[numeric] = bars[numeric].apply(pd.to_numeric, errors="raise")
    return bars.set_index("open_time").sort_index()

expected_index = pd.date_range(START, END - pd.Timedelta(minutes=5), freq="5min")
if DATA_SOURCE == "binance":
    bars = load_binance_5m()
    true_jumps = None
else:
    bars, true_jumps = sb.synthetic_5m(expected_index, trends=True, jumps=True)
    print("SYNTHETIC data: results below say nothing about BTC.")

assert bars.index.is_unique
assert bars.index.equals(expected_index), "Unexpected candle coverage"
assert (bars[["open", "high", "low", "close"]] > 0).all().all()
print("Rows:", len(bars), bars.index.min(), "->", bars.index.max())

## 2. Fit and compute bands

**Index convention:** the index is the candle **open** time, as in E0.2a; values at a row are known at that candle's close.

Only the jump filter has fitted parameters (calm/jump variance and persistence, by MLE on **TRAIN rows only**). The center speeds are design choices (`N`), not fitted. That's deliberate: on 5m BTC the likelihood-optimal level tracker is almost the price itself, which gives no smoothing at all.

In [ ]:
close = bars["close"]
idx = close.index
logp = np.log(close.values)
h_prev = sb.return_variance(logp)            # EWMA (1 day) variance of 5m returns, known at t-1

train = (idx >= EVAL_START) & (idx < TRAIN_END)
test = (idx >= TRAIN_END) & (idx < EVAL_END)

jump_params = sb.fit_jump_filter(logp, h_prev, train)
print({k: round(v, 4) for k, v in jump_params.items()})
print(f"calm-state expected duration {1/(1-jump_params['p00']):.1f} bars, "
      f"jump-state {1/(1-jump_params['p11']):.1f} bars")

methods = {
    "BB(20)":      sb.band_bb(close, 20),
    "BB(10)":      sb.band_bb(close, 10),
    "BB(5)":       sb.band_bb(close, 5),
    "EWMA(10)":    sb.band_ewma(close, 10),
    "LLT(20)":     sb.band_llt(close, 20, h_prev=h_prev),
    "JA-LLT(20)":  sb.band_ja_llt(close, 20, jump_params, h_prev=h_prev, robust_scale=False),
    "JA-LLT(20)r": sb.band_ja_llt(close, 20, jump_params, h_prev=h_prev, robust_scale=True),
}
COLORS = dict(zip(methods, ["#2a6fdb", "#7aa6ef", "#b9cdf2", "#8a8a8a", "#e08a1e", "#1f9e6e", "#7a3fb8"]))
p_jump = methods["JA-LLT(20)"]["p_jump"]
print("share of TEST bars with P(jump) > 0.5:", round((p_jump[test] > 0.5).mean(), 4))

## 3. Containment vs band width (TEST)

Each curve sweeps k. **Higher and further left is better:** more closes inside for a narrower band. If one curve sits above another everywhere, that method is a strictly better envelope.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
for name, d in methods.items():
    cc = sb.containment_curve(d.z[test], d.width_bps_per_k[test])
    ax.plot(cc.mean_width_bps, 100 * cc.containment, lw=2, color=COLORS[name], label=name)
ax.axhline(100 * TARGET_CONTAINMENT, color="#999", lw=1, ls="--")
ax.set_xlim(0, 200); ax.set_ylim(70, 100)
ax.set_xlabel("Mean full band width (bps of price)")
ax.set_ylabel("Closes inside the band published at t−1 (%)")
ax.set_title("Containment vs width, TEST (H2 2025)")
ax.grid(alpha=.2); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 4. Calibrated comparison

k is set on TRAIN for 95% containment, then applied to TEST.

- `width_bps`: the price you pay for containment.
- `median_lag`: consecutive bars outside the band starting at a shock bar. Lower = faster reaction.
- `post_cont`: share of the 12 bars after a shock that are inside the band.

In [ ]:
shocks = sb.shock_times(close, h_prev, SHOCK_SIGMA)
shocks_test = shocks[(shocks >= TRAIN_END) & (shocks < EVAL_END)]
print("TEST shocks:", len(shocks_test))

K = {}
rows = []
for name, d in methods.items():
    k = sb.calibrate_k(d.z[train], TARGET_CONTAINMENT); K[name] = k
    zt = d.z[test]
    rs = sb.reaction_stats(zt, k, shocks_test)
    rows.append(dict(method=name, k=round(k, 2),
                     train_cont=round((d.z[train].abs() <= k).mean(), 4),
                     test_cont=round((zt.abs() <= k).mean(), 4),
                     width_bps=round(k * d.width_bps_per_k[test].mean(), 1),
                     median_lag=rs["median_lag_bars"], mean_lag=round(rs["mean_lag_bars"], 2),
                     post_cont=round(rs["post_containment"], 3)))
calib = pd.DataFrame(rows).set_index("method")
display(calib)

## 5. Do band breaks predict anything? (TEST, net of fees)

Event = **first close outside** the calibrated band (+1 above, −1 below). The position is taken at that close in the break direction and held for h bars. Events within h bars of the previous event are dropped, so samples don't overlap.

- `mean_bps > 0` → continuation (breakout). `mean_bps < 0` → mean reversion (fade it instead).
- `net_bps` subtracts the round-trip cost. A real edge needs |mean| clearly above cost **and** |t| > ~3.
- `JA regime` is the jump filter's own signal: P(jump) crosses 0.5, direction = sign of z.

In [ ]:
rows = []
for h in HORIZONS:
    for name, d in methods.items():
        ev = sb.cross_events(d.z, K[name])[test]
        r = sb.event_returns(close[test], ev, h, COST_BPS)
        rows.append(dict(h=h, signal=name, **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items()}))
    ev = sb.regime_events(p_jump, methods["JA-LLT(20)"].z)[test]
    r = sb.event_returns(close[test], ev, h, COST_BPS)
    rows.append(dict(h=h, signal="JA regime", **{k: (round(v, 2) if isinstance(v, float) else v) for k, v in r.items()}))
# unconditional baseline: always long, non-overlapping
for h in HORIZONS:
    lr = np.log(close[test].values); fr = 1e4 * (lr[h::h] - lr[:-h:h])
    rows.append(dict(h=h, signal="baseline long", n=len(fr), mean_bps=round(fr.mean(), 2),
                     t_stat=round(fr.mean() / (fr.std(ddof=1) / np.sqrt(len(fr))), 2), hit=round((fr > 0).mean(), 3)))
breaks = pd.DataFrame(rows).set_index(["h", "signal"])
display(breaks)

## 6. November 2025 visual check

All bands are drawn **as published at t−1**, i.e. the level the bot could actually trade against during bar t. The lower panel shows P(jump) from the switching filter.

In [ ]:
def plot_window(names=("BB(20)", "JA-LLT(20)"), start=NOV_START, end=NOV_END):
    sl = slice(start, end)
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                                  gridspec_kw={"height_ratios": [4, 1]})
    c = close.loc[sl]
    ax.plot(c.index, c, color="black", lw=1, label="BTC close")
    for name in names:
        d = methods[name].loc[sl]; k = K[name]
        up = d.center_fc * np.exp(k * d.scale_log); lo = d.center_fc * np.exp(-k * d.scale_log)
        ax.plot(d.index, d.center_fc, color=COLORS[name], lw=1, ls=":")
        ax.plot(d.index, up, color=COLORS[name], lw=1.2, label=f"{name}  k={k:.2f}")
        ax.plot(d.index, lo, color=COLORS[name], lw=1.2)
    if true_jumps is not None:
        for t in true_jumps[(true_jumps >= pd.Timestamp(start, tz="UTC")) & (true_jumps < pd.Timestamp(end, tz="UTC"))]:
            ax.axvline(t, color="#c33", lw=.6, alpha=.6)
    ax.set_ylabel("BTCUSDT"); ax.grid(alpha=.2); ax.legend(loc="upper right", frameon=False)
    ax.set_title("Bands published at t−1 (calibrated to 95% on H1 2025)")
    pj = methods["JA-LLT(20)"]["p_jump"].loc[sl]
    ax2.fill_between(pj.index, 0, pj, step="post", color=COLORS["JA-LLT(20)"], alpha=.5, lw=0)
    ax2.set_ylim(0, 1); ax2.set_ylabel("P(jump)"); ax2.grid(alpha=.2)
    ax2.set_xlabel("Candle open (UTC)")
    plt.tight_layout(); return fig

plot_window(); plt.show()
plot_window(("BB(10)", "LLT(20)")); plt.show()

## 7. Reading the results

- **Section 3 / 4:** if `JA-LLT` reaches 95% containment with a clearly narrower band than `BB(20)` / `BB(10)` and a lower lag, the state-space part is doing its job **as an envelope**.
- **Check the ladder:** if `EWMA(10)` already gets most of the gain, the slope and jump parts add little. Prefer the simpler model.
- **Section 5 is the one that matters for the bot.** A faster band absorbs part of each breakout by design. Compare break-event returns across methods: if JA-LLT bands are tighter but their breaks carry less forward return than BB breaks, that's the trade-off showing up, not a bug.
- `train_cont` vs `test_cont`: a big drop means the calibration doesn't carry over (regime change, fat tails).
- Results on `DATA_SOURCE="synthetic"` only check the code. With `trends=False` in `synthetic_5m` every signal must show ~0 edge; any edge there means look-ahead leakage.